# [Bonus Tasks](svm_segm.ipynb)

In [12]:
import numpy as np
import matplotlib.pyplot as plt

import scipy.ndimage as ndi
import skimage.morphology as morph
import skimage.util
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from skimage.util import view_as_blocks

In [13]:
def blend_result(img, seg, border_radius=1, interior_opacity=1, interior_opacity_decay=0.9, color=(0,1,0)):
    img  = np.dstack([img] * 3).copy()
    img -= img.min()
    img /= img.max()
    selem  = morph.disk(border_radius)
    seg_bd = np.logical_xor(morph.binary_dilation(seg, selem), morph.binary_erosion(seg, selem))
    mask_decay = ndi.distance_transform_edt(seg)
    for i in range(3):
        opacity = interior_opacity / pow(1 + mask_decay[seg], interior_opacity_decay)
        img[:,:,i][seg] = color[i] * opacity + (1 - opacity) * img[:,:,i][seg]
        img[:,:,i][seg_bd] = color[i]
    return img

In [14]:
patch_sizes = (32, 32)

---

**[Task 1.1.]()** Implement `create_data_matrix`:

In [15]:
def create_data_matrix(img):
    n = 32                                   

    h = (img.shape[0] // n) * n
    w = (img.shape[1] // n) * n
    img = img[:h, :w]

    blocks = view_as_blocks(img, (n, n))

    rows = []
    for r in range(blocks.shape[0]):        
        for c in range(blocks.shape[1]):     
            patch = blocks[r, c]             #spezifische 32 x 32 Kachel
            rows.append(patch.flatten())     # alle Einträge in einen Vektor einreihen

    X = np.array(rows) # Liste aus Vektoren (rows) zu Matrix umwandeln (ein Vektor (Kacheleinträge), pro Zeile)
    return X

**[Task 1.2.]()** Implement `create_gt_labels_vector`:

In [16]:
def create_gt_labels_vector(gt):
    n = 32

    h = (gt.shape[0] // n) * n
    w = (gt.shape[1] // n) * n
    gt = gt[:h, :w]     # korresspondierendes Bild nur mit 1en für Forground und 0en für Background

    blocks = view_as_blocks(gt, (n, n))

    labels = []
    for r in range(blocks.shape[0]):             
        for c in range(blocks.shape[1]):         
            patch = blocks[r, c]            # wählt wieder 32 x 32 Kachel aus
            anteil = patch.mean()           # patch.mean gibt den Anteil an 1en (Forground) an       

            if anteil > 0.5:
                labels.append(1)
            elif anteil == 0:
                labels.append(-1)
            else:
                labels.append(0)            # Jede Kachel bekommt den Wert 1 (wenn mehr Forground als Background), -1 wenn umgekehrt oder 0 wenn 50/50

    y = np.array(labels)
    return y

**[Task 1.3.]()** Create the SVM classifier:

In [17]:
clf = make_pipeline(StandardScaler(), SVC(class_weight='balanced', gamma=0.1))

**[Task 1.3 (a).]()** Create the data matrices for the images `dna-33` and `dna-44`:

In [18]:
dna33_img = plt.imread('data/NIH3T3/im/dna-33.png')
dna44_img = plt.imread('data/NIH3T3/im/dna-44.png')

X33 = create_data_matrix(dna33_img)
X44 = create_data_matrix(dna44_img)

**[Task 1.3 (b).]()** Create the corresponding ground truth label vectors:

In [22]:
dna33_gt = plt.imread('data/NIH3T3/gt/33.png')
dna44_gt = plt.imread('data/NIH3T3/gt/44.png')

y33 = create_gt_labels_vector(dna33_gt)
y44 = create_gt_labels_vector(dna44_gt)

#Kontrolle:

print(X33.shape, y33.shape)
print(X44.shape, y44.shape)

(1344, 1024) (1344,)
(1344, 1024) (1344,)


**[Task 1.3 (c).]()** Create the *combined* data matrices and ground truth label vectors:

**[Task 1.3 (d).]()** Train the classifier using the data matrix and label vectors from above:

**[Task 1.4.]()** Implement the function `predict_image`:

Test your implementation:

In [ ]:
img = plt.imread(f'data/NIH3T3/im/dna-0.png')
seg = predict_image(img)

plt.figure()
plt.imshow(blend_result(img, seg))

**[Task 1.5.]()** Perform batch processing: